# 03. Discriminative Intent Description 생성 (Step E)

**목적**: 초기 IntentSet의 907개 의도 각각에 대해, 유사 의도와 구별되는 판별적 설명(Discriminative Description)을 생성한다. 이 설명은 이후 Loop 단계에서 LLM이 저신뢰 쿼리를 분류할 때 프롬프트에 삽입되어 분류 정확도를 높인다.

**방법론**: FCSLM 논문의 아이디어를 적용.
1. 라벨 임베딩 기반 FAISS 인덱스로 의도별 Top-5 유사 의도 검색 (geodesic < 1.0 필터링)
2. Gemini API로 대상 의도와 유사 의도를 비교하여 구조화된 JSON 설명 생성

**산출물**:
- `data/processed/intent_descriptions.parquet` — 907개 의도의 Discriminative Description

**예상 비용**: ~₩160 / 잔여 예산 ~₩43,800

**예상 소요**: ~5분

## 0. 환경 설정

In [1]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')
print(f'작업 디렉토리: {os.getcwd()}')

!pip install -q google-genai sentence-transformers faiss-cpu

import json
import time
import unicodedata
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.9 MB/s eta 0:00:00


## 0-1. 경로 및 하이퍼파라미터 설정

In [2]:
# === 경로 설정 ===
BASE_DIR = Path('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

PATH_CONFIG = {
    'processed': BASE_DIR / 'data' / 'processed',
    'checkpoints': BASE_DIR / 'checkpoints',
    'embeddings': BASE_DIR / 'embeddings',
    'vector_db': BASE_DIR / 'vector_db',
}

# 입력
PATH_CONFIG['initial_intentset'] = PATH_CONFIG['processed'] / 'initial_intentset.parquet'
PATH_CONFIG['intentset_summary'] = PATH_CONFIG['processed'] / 'intentset_summary.parquet'

# 산출물
PATH_CONFIG['intent_descriptions'] = PATH_CONFIG['processed'] / 'intent_descriptions.parquet'
PATH_CONFIG['desc_ckpt_dir'] = PATH_CONFIG['checkpoints'] / 'descriptions'
PATH_CONFIG['label_faiss_index'] = PATH_CONFIG['vector_db'] / 'label_embeddings.index'

# 디렉토리 생성
PATH_CONFIG['desc_ckpt_dir'].mkdir(parents=True, exist_ok=True)

# === 하이퍼파라미터 ===
CONFIG = {
    # 임베딩
    'embed_model_name': 'snunlp/KR-SBERT-V40K-klueNLI-augSTS',

    # 유사 의도 검색
    'desc_top_k': 5,                     # FAISS Top-K
    'desc_geodesic_max': 1.0,            # geodesic distance 필터 상한

    # Gemini API
    'gemini_model': 'gemini-2.5-flash-lite',
    'gemini_temperature': 0.2,
    'gemini_max_output_tokens': 512,
    'gemini_rate_limit_delay': 0.25,
    'gemini_retry_max': 3,
    'gemini_retry_delay': 5,

    # 배치 체크포인트
    'desc_batch_size': 100,
}

for key in ['initial_intentset', 'intentset_summary']:
    marker = '✓' if PATH_CONFIG[key].exists() else '✗'
    print(f'  {marker} {key}: {PATH_CONFIG[key]}')

print(f'\n하이퍼파라미터:')
print(f'  Top-K: {CONFIG["desc_top_k"]}, geodesic 필터: < {CONFIG["desc_geodesic_max"]}')

  ✓ initial_intentset: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/initial_intentset.parquet
  ✓ intentset_summary: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/intentset_summary.parquet

하이퍼파라미터:
  Top-K: 5, geodesic 필터: < 1.0


## 0-2. Gemini API 초기화

In [3]:
# === Gemini API 초기화 ===
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

test_response = client.models.generate_content(
    model=CONFIG['gemini_model'],
    contents='테스트입니다. "ok"라고만 답하세요.',
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=10),
)
print(f'Gemini API 연결 성공: {test_response.text}')

Gemini API 연결 성공: ok


## 0-3. 유틸리티 함수 정의

In [4]:
# === 유틸리티 함수 ===

def nfc(text: str) -> str:
    return unicodedata.normalize('NFC', text)


def call_gemini(
    prompt: str,
    system_instruction: str,
) -> str:
    for attempt in range(CONFIG['gemini_retry_max']):
        try:
            response = client.models.generate_content(
                model=CONFIG['gemini_model'],
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=CONFIG['gemini_temperature'],
                    max_output_tokens=CONFIG['gemini_max_output_tokens'],
                    response_mime_type='application/json',
                ),
            )
            time.sleep(CONFIG['gemini_rate_limit_delay'])
            return response.text.strip()
        except Exception as e:
            error_str = str(e)
            if '429' in error_str or 'RESOURCE_EXHAUSTED' in error_str:
                wait = CONFIG['gemini_retry_delay'] * (attempt + 1) * 2
                print(f'  [Rate limit] {wait}s 대기 ({attempt+1}/{CONFIG["gemini_retry_max"]})')
                time.sleep(wait)
            else:
                wait = CONFIG['gemini_retry_delay'] * (attempt + 1)
                print(f'  [Error] {error_str[:80]}... ({attempt+1}/{CONFIG["gemini_retry_max"]})')
                time.sleep(wait)
    return ''


print('유틸리티 함수 정의 완료')

유틸리티 함수 정의 완료


## 1. IntentSet 요약 데이터 로드
`02_clustering_dial_in`에서 생성한 클러스터 요약 데이터를 로드한다.

In [7]:
# === IntentSet 로드 ===

def load_intentset_summary() -> pd.DataFrame:
    df = pd.read_parquet(PATH_CONFIG['intentset_summary'])
    df['intent_label'] = df['intent_label'].apply(nfc)
    df['sample_sentences'] = df['sample_sentences'].apply(nfc)
    print(f'IntentSet 요약 로드: {len(df)} 클러스터')
    print(f'  총 문장: {df["size"].sum():,}')
    print(f'  고유 라벨: {df["intent_label"].nunique()}')
    return df


summary_df = load_intentset_summary()
summary_df.head(3)

IntentSet 요약 로드: 907 클러스터
  총 문장: 33,341
  고유 라벨: 907


,cluster_id,intent_label,size,sample_sentences
0,0,문의-공항픽업서비스,274,"[""문의 — 공항 픽업 서비스"", ""문의 — 공항 픽업 서비스 이용 지점"", ""문의..."
1,1,요청-카드분실재발급,35,"[""등록 요청 — 신규 발급 카드"", ""요청 — 카드 결제 정보 정정 및 재시도"",..."
2,2,문의-추가비용,30,"[""문의 — 액티비티 선택 가능 여부 및 추가 비용"", ""문의 — 추가 인원 관련 ..."


## 2. 라벨 임베딩 생성 및 FAISS 인덱스 구축
907개 의도 라벨의 임베딩을 생성하고, FAISS Inner Product 인덱스를 구축하여 유사 의도 검색에 사용한다.

**체크포인트**: `vector_db/label_embeddings.index`

In [8]:
# === 라벨 임베딩 + FAISS 인덱스 ===

def build_label_faiss_index(labels: list) -> tuple:
    """
    라벨 임베딩을 생성하고 FAISS Inner Product 인덱스를 구축한다.
    L2 정규화된 벡터에 대한 Inner Product = cosine similarity.

    반환: (label_embeddings, faiss_index)
    """
    emb_path = PATH_CONFIG['embeddings'] / 'label_embeddings.npy'
    index_path = PATH_CONFIG['label_faiss_index']

    # 임베딩 로드 또는 생성
    if emb_path.exists():
        print(f'라벨 임베딩 캐시 로드: {emb_path}')
        label_embs = np.load(emb_path)
    else:
        print(f'라벨 임베딩 생성 중... ({len(labels)}개)')
        model = SentenceTransformer(CONFIG['embed_model_name'])
        label_embs = model.encode(labels, normalize_embeddings=True)
        np.save(emb_path, label_embs)
        print(f'라벨 임베딩 저장: {emb_path}')

    label_embs = label_embs.astype(np.float32)
    d = label_embs.shape[1]
    print(f'  shape: {label_embs.shape}, L2 norm: {np.linalg.norm(label_embs[0]):.4f}')

    # FAISS Inner Product 인덱스
    index = faiss.IndexFlatIP(d)
    index.add(label_embs)
    print(f'FAISS 인덱스 구축 완료: {index.ntotal}개 벡터')

    return label_embs, index


labels = summary_df['intent_label'].tolist()
label_embs, faiss_index = build_label_faiss_index(labels)

라벨 임베딩 캐시 로드: /content/drive/MyDrive/5stone_experiment/1_clustering_test/embeddings/label_embeddings.npy
  shape: (907, 768), L2 norm: 1.0000
FAISS 인덱스 구축 완료: 907개 벡터


## 3. 유사 의도 검색
각 의도에 대해 Top-5 유사 의도를 FAISS로 검색하고, geodesic distance < 1.0 필터를 적용한다.

In [9]:
# === 유사 의도 검색 ===

def find_similar_intents(
    label_embs: np.ndarray,
    faiss_index: faiss.IndexFlatIP,
    labels: list,
    top_k: int = 5,
    geodesic_max: float = 1.0,
) -> dict:
    """
    각 의도에 대해 Top-K 유사 의도를 검색하고, geodesic distance 필터를 적용한다.
    자기 자신은 제외한다.

    반환: {cluster_id: [(similar_cluster_id, label, geodesic_dist), ...]}
    """
    # Top-K+1 검색 (자기 자신 포함)
    search_k = top_k + 1
    similarities, indices = faiss_index.search(label_embs, search_k)

    similar_map = {}
    filter_stats = {'total_candidates': 0, 'after_filter': 0}

    for i in range(len(labels)):
        neighbors = []
        for rank in range(search_k):
            j = indices[i][rank]
            if j == i:  # 자기 자신 제외
                continue

            # cosine similarity → geodesic distance
            cos_sim = float(np.clip(similarities[i][rank], -1.0, 1.0))
            geodesic = float(np.arccos(cos_sim))

            filter_stats['total_candidates'] += 1
            if geodesic < geodesic_max:
                neighbors.append((int(j), labels[j], round(geodesic, 4)))
                filter_stats['after_filter'] += 1

        similar_map[i] = neighbors

    # 통계
    n_with_similar = sum(1 for v in similar_map.values() if v)
    avg_similar = np.mean([len(v) for v in similar_map.values()])
    n_isolated = sum(1 for v in similar_map.values() if not v)

    print(f'=== 유사 의도 검색 결과 ===')
    print(f'  총 의도: {len(labels)}')
    print(f'  유사 의도 있음: {n_with_similar} ({n_with_similar/len(labels)*100:.1f}%)')
    print(f'  유사 의도 없음 (고립): {n_isolated}')
    print(f'  평균 유사 의도 수: {avg_similar:.1f}')
    print(f'  필터 전 후보: {filter_stats["total_candidates"]:,} → 필터 후: {filter_stats["after_filter"]:,}')

    # 유사도 분포 샘플
    print(f'\n--- 유사 의도 샘플 (처음 5개) ---')
    for i in range(min(5, len(labels))):
        print(f'  [{labels[i]}]')
        for j, lbl, dist in similar_map[i][:3]:
            print(f'    ↔ {lbl} (geodesic={dist})')

    return similar_map


similar_map = find_similar_intents(
    label_embs, faiss_index, labels,
    top_k=CONFIG['desc_top_k'],
    geodesic_max=CONFIG['desc_geodesic_max'],
)

=== 유사 의도 검색 결과 ===
  총 의도: 907
  유사 의도 있음: 903 (99.6%)
  유사 의도 없음 (고립): 4
  평균 유사 의도 수: 4.9
  필터 전 후보: 4,535 → 필터 후: 4,474

--- 유사 의도 샘플 (처음 5개) ---
  [문의-공항픽업서비스]
    ↔ 문의-공항픽업예약 (geodesic=0.4463)
    ↔ 문의-숙박예약및이동픽업 (geodesic=0.7575)
    ↔ 문의-항공편스케줄 (geodesic=0.7809)
  [요청-카드분실재발급]
    ↔ 요청-카드재발급 (geodesic=0.485)
    ↔ 확인요청-카드재발급 (geodesic=0.4873)
    ↔ 요청-카드사용내역발급 (geodesic=0.5534)
  [문의-추가비용]
    ↔ 문의-예약비용 (geodesic=0.5456)
    ↔ 문의-숙박비용 (geodesic=0.672)
    ↔ 문의-소요시간 (geodesic=0.7393)
  [문의-캐시백]
    ↔ 확인요청-캐시백처리 (geodesic=0.6173)
    ↔ 문의-카드혜택 (geodesic=0.6334)
    ↔ 문의-제휴카드할인 (geodesic=0.6417)
  [확인요청-할부잔여기간]
    ↔ 확인요청-할부잔액 (geodesic=0.4212)
    ↔ 문의-할부잔여금액확인 (geodesic=0.5394)
    ↔ 확인요청-할부금정보 (geodesic=0.5553)


## 4. Discriminative Description 생성
Gemini API로 각 의도의 판별적 설명을 구조화 JSON으로 생성한다.

**출력 형식**:
```json
{
  "intent": "요청-환불",
  "definition": "이미 완료된 결제에 대한 금액 반환을 요청하는 의도",
  "differs_from": [
    {"intent": "요청-결제취소", "distinction": "결제가 완료되기 전에 취소하는 것과 구별"},
    ...
  ]
}
```

**체크포인트**: `checkpoints/descriptions/desc_batch_*.json` (100건 단위)

In [5]:
# === Discriminative Description 프롬프트 및 생성 ===

DESC_SYSTEM = """당신은 고객 상담 의도 분류 시스템을 설계하는 전문가입니다.

## 작업
주어진 의도(intent)에 대해, 유사 의도와 구별되는 판별적 설명(Discriminative Description)을 생성하세요.
이 설명은 분류 모델이 혼동하기 쉬운 의도 쌍을 구별하는 데 사용됩니다.

## 규칙
1. definition: 이 의도가 무엇인지 한 문장으로 정의하세요.
2. differs_from: 각 유사 의도와 어떤 점에서 다른지 구체적으로 설명하세요.
3. 고객 상담 맥락에서 실제로 혼동될 수 있는 차이점에 집중하세요.
4. 유사 의도가 제공되지 않은 경우, differs_from은 빈 배열로 출력하세요.

## 출력 형식 (JSON)
{
  "intent": "행위-목적",
  "definition": "이 의도의 정의",
  "differs_from": [
    {"intent": "유사의도1", "distinction": "구별 기준 설명"},
    {"intent": "유사의도2", "distinction": "구별 기준 설명"}
  ]
}"""


def build_desc_prompt(
    target_label: str,
    sample_sentences: list,
    similar_intents: list,
) -> str:
    """
    Discriminative Description 생성 프롬프트를 구성한다.
    """
    prompt_parts = []
    prompt_parts.append(f'대상 의도: {target_label}')
    prompt_parts.append(f'소속 문장 예시: {json.dumps(sample_sentences[:5], ensure_ascii=False)}')

    if similar_intents:
        similar_labels = [lbl for _, lbl, _ in similar_intents]
        prompt_parts.append(f'유사 의도 (혼동 가능): {json.dumps(similar_labels, ensure_ascii=False)}')
    else:
        prompt_parts.append('유사 의도: 없음 (고립된 의도)')

    return '\n'.join(prompt_parts)


def parse_desc_response(response_text: str, target_label: str) -> dict:
    """
    Gemini 응답 JSON을 파싱하여 description dict를 반환한다.
    파싱 실패 시 최소 구조를 반환.
    """
    if not response_text:
        return {
            'intent': target_label,
            'definition': '',
            'differs_from': [],
            'is_fallback': True,
        }
    try:
        data = json.loads(response_text)
        data['is_fallback'] = False
        return data
    except json.JSONDecodeError:
        return {
            'intent': target_label,
            'definition': '',
            'differs_from': [],
            'is_fallback': True,
            'raw_response': response_text[:300],
        }


print('프롬프트 및 파서 정의 완료')

프롬프트 및 파서 정의 완료


### 4-1. 프롬프트 검증 (샘플 테스트)
3건으로 프롬프트 품질을 확인한다.

In [9]:
# === 프롬프트 검증 ===

def test_desc_prompt(summary_df: pd.DataFrame, similar_map: dict, n_samples: int = 3) -> None:
    sample_ids = [0, len(summary_df)//2, len(summary_df)-1]

    for cid in sample_ids[:n_samples]:
        row = summary_df.iloc[cid]
        label = row['intent_label']
        samples = json.loads(row['sample_sentences'])
        similars = similar_map.get(cid, [])

        print(f'\n{"="*60}')
        print(f'cluster_id={cid} | {label} | size={row["size"]} | 유사 의도: {len(similars)}개')
        print(f'{"="*60}')

        prompt = build_desc_prompt(label, samples, similars)
        response = call_gemini(prompt, DESC_SYSTEM)
        desc = parse_desc_response(response, label)

        print(f'  definition: {desc.get("definition", "N/A")}')
        for d in desc.get('differs_from', []):
            print(f'  ↔ {d.get("intent", "?")}: {d.get("distinction", "?")}')


test_desc_prompt(summary_df, similar_map)


cluster_id=0 | 문의-공항픽업서비스 | size=274 | 유사 의도: 5개
  definition: 공항 픽업 서비스 자체에 대한 전반적인 정보, 이용 방법, 절차, 지점, 또는 미이용 시 대안 등에 대해 문의하는 의도입니다.
  ↔ 문의-공항픽업예약: 이 의도는 공항 픽업 서비스의 예약 절차, 예약 가능 여부, 예약 변경/취소 등에 초점을 맞추는 반면, '문의-공항픽업서비스'는 서비스 자체의 일반적인 정보나 이용 가능 여부 등을 문의합니다.
  ↔ 문의-숙박예약및이동픽업: 이 의도는 숙박 예약과 연계된 이동 픽업 서비스에 대한 문의이며, '문의-공항픽업서비스'는 숙박 예약과 무관하게 공항에서 목적지까지의 픽업 서비스 자체에 대한 문의입니다.
  ↔ 문의-항공편스케줄: 이 의도는 항공편의 출발, 도착 시간, 지연 여부 등 항공편 자체의 스케줄에 대한 문의이며, '문의-공항픽업서비스'는 공항 픽업 서비스 이용과 관련된 문의입니다.
  ↔ 문의-발리공항서비스: 이 의도는 특정 지역(발리)의 공항에서 제공하는 전반적인 서비스에 대한 문의이며, '문의-공항픽업서비스'는 공항 픽업 서비스라는 특정 서비스 종류에 대한 문의입니다.
  ↔ 문의-싱가포르항공: 이 의도는 싱가포르항공이라는 특정 항공사에 대한 문의이며, '문의-공항픽업서비스'는 공항 픽업 서비스라는 특정 서비스 종류에 대한 문의입니다.

cluster_id=453 | 견적요청-숙박 | size=37 | 유사 의도: 5개
  definition: 특정 숙박 시설에 대한 가격 및 포함 내역 등 견적 정보를 요청하는 의도입니다.
  ↔ 견적요청-숙소: '견적요청-숙박'은 특정 숙박 시설(예: 파드마 우붓, 아야나 스가라)을 명시하며 견적을 요청하는 반면, '견적요청-숙소'는 특정 숙박 시설을 명시하지 않고 일반적인 숙소에 대한 견적을 요청할 수 있습니다.
  ↔ 요청-숙소예약: '견적요청-숙박'은 숙박에 대한 가격 정보를 얻는 것이 목적이지만, '요청-숙소예약'은 숙박 예약을 확정하려는 의도입니다.


### 4-2. 전체 생성 실행
907개 의도에 대해 Discriminative Description을 생성한다. 100건 단위로 체크포인트를 저장한다.

**재개 로직**: 체크포인트에서 이미 생성된 cluster_id를 건너뛴다.

In [10]:
# === 전체 생성 ===

def get_processed_desc_ids() -> set:
    """체크포인트에서 이미 처리된 cluster_id 집합을 반환한다."""
    ckpt_dir = PATH_CONFIG['desc_ckpt_dir']
    processed = set()
    for f in ckpt_dir.glob('desc_batch_*.json'):
        try:
            with open(f, 'r', encoding='utf-8') as fp:
                batch = json.load(fp)
            for item in batch:
                processed.add(item['cluster_id'])
        except Exception as e:
            print(f'  [경고] 체크포인트 읽기 실패: {f.name}: {e}')
    return processed


def run_description_generation(
    summary_df: pd.DataFrame,
    similar_map: dict,
) -> list:
    """
    전체 의도에 대해 Discriminative Description을 생성한다.
    """
    processed_ids = get_processed_desc_ids()
    all_ids = set(range(len(summary_df)))
    remaining_ids = sorted(all_ids - processed_ids)

    print(f'전체: {len(all_ids)} | 처리 완료: {len(processed_ids)} | 남은: {len(remaining_ids)}')

    if not remaining_ids:
        print('모든 의도가 이미 처리됨. 체크포인트에서 병합합니다.')
        return merge_desc_checkpoints()

    batch_results = []
    batch_counter = len(list(PATH_CONFIG['desc_ckpt_dir'].glob('desc_batch_*.json')))
    fail_count = 0

    for cid in tqdm(remaining_ids, desc='Description 생성'):
        row = summary_df.iloc[cid]
        label = row['intent_label']
        samples = json.loads(row['sample_sentences'])
        similars = similar_map.get(cid, [])

        prompt = build_desc_prompt(label, samples, similars)
        response = call_gemini(prompt, DESC_SYSTEM)
        desc = parse_desc_response(response, label)
        desc['cluster_id'] = cid
        desc['n_similar'] = len(similars)

        if desc.get('is_fallback'):
            fail_count += 1

        batch_results.append(desc)

        # 배치 체크포인트
        if len(batch_results) >= CONFIG['desc_batch_size']:
            ckpt_path = PATH_CONFIG['desc_ckpt_dir'] / f'desc_batch_{batch_counter:04d}.json'
            with open(ckpt_path, 'w', encoding='utf-8') as f:
                json.dump(batch_results, f, ensure_ascii=False, indent=2)
            batch_counter += 1
            batch_results = []

    # 남은 배치 저장
    if batch_results:
        ckpt_path = PATH_CONFIG['desc_ckpt_dir'] / f'desc_batch_{batch_counter:04d}.json'
        with open(ckpt_path, 'w', encoding='utf-8') as f:
            json.dump(batch_results, f, ensure_ascii=False, indent=2)

    print(f'\n생성 완료. Fallback: {fail_count}건')
    return merge_desc_checkpoints()


def merge_desc_checkpoints() -> list:
    """체크포인트를 병합하여 전체 description 리스트를 반환한다."""
    ckpt_dir = PATH_CONFIG['desc_ckpt_dir']
    all_descs = []
    for f in sorted(ckpt_dir.glob('desc_batch_*.json')):
        with open(f, 'r', encoding='utf-8') as fp:
            all_descs.extend(json.load(fp))

    # cluster_id 기준 중복 제거
    seen = set()
    unique = []
    for d in all_descs:
        cid = d['cluster_id']
        if cid not in seen:
            seen.add(cid)
            unique.append(d)

    print(f'체크포인트 병합: {len(unique)}개 description')
    return unique


all_descriptions = run_description_generation(summary_df, similar_map)

전체: 907 | 처리 완료: 753 | 남은: 154


Description 생성:   0%|          | 0/154 [00:00<?, ?it/s]


생성 완료. Fallback: 79건
체크포인트 병합: 907개 description


## 5. 결과 검증 및 통계

In [11]:
# === 결과 검증 ===

def validate_descriptions(descriptions: list) -> dict:
    stats = {}

    n_total = len(descriptions)
    n_fallback = sum(1 for d in descriptions if d.get('is_fallback'))
    n_with_diffs = sum(1 for d in descriptions if d.get('differs_from'))
    n_isolated = sum(1 for d in descriptions if not d.get('differs_from'))

    print(f'=== Description 통계 ===')
    print(f'  총 의도: {n_total}')
    print(f'  Fallback: {n_fallback} ({n_fallback/n_total*100:.1f}%)')
    print(f'  differs_from 있음: {n_with_diffs} ({n_with_diffs/n_total*100:.1f}%)')
    print(f'  고립 (differs_from 없음): {n_isolated}')

    # definition 길이 분포
    def_lens = [len(d.get('definition', '')) for d in descriptions if not d.get('is_fallback')]
    if def_lens:
        print(f'\n--- definition 길이 (글자 수) ---')
        print(f'  평균: {np.mean(def_lens):.1f}, 중앙값: {np.median(def_lens):.0f}')
        print(f'  최소: {min(def_lens)}, 최대: {max(def_lens)}')

    # differs_from 개수 분포
    diff_counts = [len(d.get('differs_from', [])) for d in descriptions]
    print(f'\n--- differs_from 개수 분포 ---')
    for n in range(max(diff_counts) + 1):
        cnt = sum(1 for c in diff_counts if c == n)
        if cnt > 0:
            print(f'  {n}개: {cnt}')

    # 샘플 출력
    print(f'\n--- 샘플 (differs_from이 풍부한 상위 3개) ---')
    rich = sorted(descriptions, key=lambda d: len(d.get('differs_from', [])), reverse=True)
    for d in rich[:3]:
        print(f'\n  [{d["intent"]}] {d.get("definition", "N/A")}')
        for diff in d.get('differs_from', []):
            print(f'    ↔ {diff.get("intent", "?")}: {diff.get("distinction", "?")}')

    return stats


desc_stats = validate_descriptions(all_descriptions)

=== Description 통계 ===
  총 의도: 907
  Fallback: 79 (8.7%)
  differs_from 있음: 824 (90.8%)
  고립 (differs_from 없음): 83

--- definition 길이 (글자 수) ---
  평균: 46.8, 중앙값: 45
  최소: 22, 최대: 101

--- differs_from 개수 분포 ---
  0개: 83
  1개: 4
  2개: 5
  3개: 5
  4개: 2
  5개: 808

--- 샘플 (differs_from이 풍부한 상위 3개) ---

  [문의-공항픽업서비스] 공항 픽업 서비스 자체에 대한 전반적인 정보, 이용 방법, 절차, 지점, 또는 미이용 시 대안 등에 대해 문의하는 의도입니다.
    ↔ 문의-공항픽업예약: 공항 픽업 서비스 자체에 대한 문의가 아니라, 해당 서비스를 '예약'하는 과정이나 방법에 초점을 맞춘 의도입니다.
    ↔ 문의-숙박예약및이동픽업: 숙박 예약과 연계된 이동 픽업 서비스에 대한 문의이며, 단순히 공항 픽업 서비스 자체에 대한 문의와는 다릅니다.
    ↔ 문의-항공편스케줄: 항공편의 출발, 도착 시간 등 스케줄 자체에 대한 문의이며, 공항 픽업 서비스와는 직접적인 관련이 없습니다.
    ↔ 문의-발리공항서비스: 특정 지역(발리)의 공항 서비스 전반에 대한 문의이며, 공항 픽업 서비스에 국한되지 않습니다.
    ↔ 문의-싱가포르항공: 싱가포르항공이라는 특정 항공사에 대한 문의이며, 공항 픽업 서비스와는 관련이 없습니다.

  [요청-카드분실재발급] 고객이 분실한 카드를 재발급받기 위해 요청하는 의도입니다.
    ↔ 요청-카드재발급: 단순 카드 재발급 요청과 달리, 반드시 '분실'이라는 사유가 명시되어야 합니다.
    ↔ 확인요청-카드재발급: 카드 재발급 가능 여부나 절차 확인을 넘어, 분실로 인한 재발급을 직접 요청하는 행위입니다.
    ↔ 요청-카드사용내역발급: 카드 사용 내역 발급 요청과는 전혀 다른 의도로, 분실된 카드의

## 6. 최종 저장
Discriminative Description을 `data/processed/intent_descriptions.parquet`에 저장한다.
`differs_from`은 JSON 문자열로 직렬화하여 parquet 호환성을 유지한다.

In [12]:
# === 최종 저장 ===

def save_descriptions(descriptions: list) -> pd.DataFrame:
    rows = []
    for d in descriptions:
        rows.append({
            'cluster_id': d['cluster_id'],
            'intent': d.get('intent', ''),
            'definition': d.get('definition', ''),
            'differs_from': json.dumps(d.get('differs_from', []), ensure_ascii=False),
            'n_similar': d.get('n_similar', 0),
            'is_fallback': d.get('is_fallback', False),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values('cluster_id').reset_index(drop=True)

    output_path = PATH_CONFIG['intent_descriptions']
    df.to_parquet(output_path, index=False)
    print(f'저장: {output_path}')
    print(f'  {len(df)} 의도, Fallback: {df["is_fallback"].sum()}')

    return df


desc_df = save_descriptions(all_descriptions)
desc_df.head(5)

저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/intent_descriptions.parquet
  907 의도, Fallback: 79


,cluster_id,intent,definition,differs_from,n_similar,is_fallback
0,0,문의-공항픽업서비스,"공항 픽업 서비스 자체에 대한 전반적인 정보, 이용 방법, 절차, 지점, 또는 미이...","[{""intent"": ""문의-공항픽업예약"", ""distinction"": ""공항 픽업...",5,False
1,1,요청-카드분실재발급,고객이 분실한 카드를 재발급받기 위해 요청하는 의도입니다.,"[{""intent"": ""요청-카드재발급"", ""distinction"": ""단순 카드 ...",5,False
2,2,문의-추가비용,서비스 이용 중 예상치 못한 추가 비용 발생 가능성에 대해 문의하는 의도입니다.,"[{""intent"": ""문의-예약비용"", ""distinction"": ""예약 시점에 ...",5,False
3,3,문의-캐시백,"고객이 카드 사용이나 이벤트 참여 등을 통해 받을 수 있는 캐시백의 적용 여부, 신...","[{""intent"": ""확인요청-캐시백처리"", ""distinction"": ""캐시백이...",5,False
4,4,확인요청-할부잔여기간,고객이 휴대폰 할부 계약의 남은 기간 또는 개월 수를 확인하고자 할 때 사용되는 의...,"[{""intent"": ""확인요청-할부잔액"", ""distinction"": ""이 의도는...",5,False


## 7. 세션 종료
세션을 자동으로 종료하고, 자원을 반환한다.

In [13]:
from google.colab import runtime
runtime.unassign()